# A multi-factor market-neutral investment strategy for New York Stock Exchange equities

This notebook implements the strategy from the paper by Georgios M. Gkolemis, Adwin Richie Lee, and Amine Roudani (2024). The strategy aims to deliver steady returns while minimizing correlation with the market by using a robust feature set combining momentum-based indicators, fundamental factors, and analyst recommendations.

**Paper Title:** A multi-factor market-neutral investment strategy for New York Stock Exchange equities
**Authors:** Georgios M. Gkolemis, Adwin Richie Lee, Amine Roudani
**Published:** 2024-12-16
**ArXiv:** https://arxiv.org/abs/2412.12350

**Abstract:** This report presents a systematic market-neutral, multi-factor investment strategy for New York Stock Exchange equities with the objective of delivering steady returns while minimizing correlation with the market. A robust feature set is integrated combining momentum-based indicators, fundamental factors, and analyst recommendations. Using various statistical tests for feature selection, the strategy identifies key drivers of equity performance and ranks stocks to build a balanced portfolio of long and short positions. Portfolio construction methods, including equally weighted, risk parity, and minimum variance beta-neutral approaches, were evaluated through rigorous backtesting. Risk parity demonstrated superior performance with a higher Sharpe ratio, lower beta, and smaller maximum drawdown compared to the Standard and Poor's 500 index. Risk parity's market neutrality, combined with its ability to maintain steady returns and mitigate large drawdowns, makes it a suitable approach for managing significant capital in equity markets.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration

# Define the universe of tickers to consider
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'FB', 'BRK-B', 'JPM', 'JNJ', 'V']

# Define parameters for the strategy
MOMENTUM_PERIOD = 12
FUNDAMENTAL_FACTORS = ['P/E Ratio', 'P/B Ratio']
ANALYST_RECOMMENDATIONS = True

# Hypothesis: A market-neutral strategy using momentum, fundamental factors, and analyst recommendations will outperform the S&P 500.
HYPOTHESIS = "A market-neutral strategy using momentum, fundamental factors, and analyst recommendations will outperform the S&P 500."

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download historical market data
data = yf.download(UNIVERSE, start='2010-01-01', end='2024-01-01', group_by='ticker')

# Compute momentum factor
momentum = data['Adj Close'].pct_change(MOMENTUM_PERIOD)

# Compute fundamental factors (example: P/E Ratio)
fundamentals = pd.DataFrame()
for ticker in UNIVERSE:
    pe_ratio = yf.Ticker(ticker).info['trailingPE']
    fundamentals[ticker] = [pe_ratio] * len(data)
fundamentals = fundamentals.T

# Combine features into a single DataFrame
features = pd.concat([momentum, fundamentals], axis=1)

# Cross-sectional normalization
features = (features - features.mean()) / features.std()

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Generate signals based on feature scores
signals = features.rank(axis=1, pct=True)

# Position sizing and portfolio construction
long_signals = signals > 0.5
short_signals = signals < 0.5

# Equally weighted portfolio
long_positions = long_signals.sum(axis=1) / long_signals.sum().sum()
short_positions = short_signals.sum(axis=1) / short_signals.sum().sum()

# Combine long and short positions
positions = long_positions - short_positions

## Phase 4 — Vectorized Backtest

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
positions_shifted = positions.shift(1)

# Calculate daily returns
daily_returns = data['Adj Close'].pct_change()

# Calculate portfolio returns
portfolio_returns = (daily_returns * positions_shifted).sum(axis=1)

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Calculate performance metrics
sharpe_ratio = np.mean(portfolio_returns) / np.std(portfolio_returns)
sortino_ratio = np.mean(portfolio_returns) / np.std(portfolio_returns[portfolio_returns < 0])
max_drawdown = (portfolio_returns.cumsum().expanding(min_periods=1).max() - portfolio_returns.cumsum()).max()
calmar_ratio = np.mean(portfolio_returns) / max_drawdown

# Print performance metrics
print(f'Sharpe Ratio: {sharpe_ratio}')
print(f'Sortino Ratio: {sortino_ratio}')
print(f'Max Drawdown: {max_drawdown}')
print(f'Calmar Ratio: {calmar_ratio}')

# Plot equity curve
plt.plot(portfolio_returns.cumsum())
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_portfolio(live_data):
    # Placeholder function for monitoring daily P&L and current positions
    daily_pnl = live_data['Adj Close'].pct_change().sum(axis=1)
    current_positions = positions.iloc[-1]
    print(f'Daily P&L: {daily_pnl}')
    print(f'Current Positions: {current_positions}')

# Example usage
monitor_portfolio(data)